# Evaluation harness

Every model in this project is scored through this notebook, so the comparisons between them are fair. It is built before any real model, and tested against predictions whose correct score is already known.

The design follows the structure of the dataset. The 35,549 focal clips are the training material. The 1,478 labelled soundscape segments are the field evaluation material, which is why the competition provides so few of them. So the question this harness answers is not "how well does the model do on held-out training data", but "how well does a model trained on clean focal audio perform on real field recordings, and how does that vary by site".

In [ ]:
import json
import numpy as np
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import roc_auc_score, average_precision_score

# load the spectrograms and metadata
specs = np.load('../data/full_specs.npy')
with open('../data/full_meta.json') as f:
    meta = json.load(f)

labels = meta['labels']
sites = meta['sites']
files = meta['files']

train = pd.read_csv('../data/train.csv')

print("segments:", len(labels))
print("spectrograms:", specs.shape)
print("sites:", sorted(set(sites)))

segments: 1478
spectrograms: (1478, 128, 313)
sites: ['S03', 'S08', 'S09', 'S13', 'S15', 'S18', 'S19', 'S22', 'S23']


## Fixing the label space

A model trained on the focal clips will output one score per species across the full training label set, not just the species that happen to appear in the soundscapes. The truth matrix therefore has to use the same full set of columns, so that predictions and truth line up position for position.

This also checks whether any species appears in the soundscape labels but not in the training data. If so, that species can never be predicted, and needs to be flagged rather than silently dropped.

In [ ]:
# load the taxonomy and determine which species have training audio and which do not
tax = pd.read_csv('../data/taxonomy.csv')
all_species = tax['primary_label'].astype(str).tolist()

trainable = set(train['primary_label'].astype(str))
untrainable = [s for s in all_species if s not in trainable]

print("full label space (taxonomy):", len(all_species))
print("species with training audio:", len(trainable))
print("species with NO training audio:", len(untrainable))

# truth matrix over the full 234-species label space
mlb = MultiLabelBinarizer(classes=all_species)
Y = mlb.fit_transform(labels)

present = Y.sum(axis=0)
heard = set(np.array(all_species)[present > 0])

print()
print("truth matrix:", Y.shape)
print("species heard in the field:", len(heard))
print("never heard in the field:", int((present == 0).sum()))
print("heard in field AND untrainable:", len(heard & set(untrainable)))

full label space (taxonomy): 234
species with training audio: 206
species with NO training audio: 28

truth matrix: (1478, 234)
species heard in the field: 75
never heard in the field: 159
heard in field AND untrainable: 28


## How much of the field audio is untrainable

Twenty-eight species are audible in the soundscapes but have no training audio at all. Counting them as a share of species is one thing, but their share of actual detections is what determines how much they distort a diversity estimate.

In [ ]:
# Analyse untrainable species
untr_idx = [all_species.index(s) for s in untrainable]
untr_hits = Y[:, untr_idx].sum()
total_hits = Y.sum()

print("segments containing at least one untrainable species:",
      int((Y[:, untr_idx].sum(axis=1) > 0).sum()), "of", Y.shape[0])
print("share of all species-detections that are untrainable:",
      round(untr_hits / total_hits, 3))

segments containing at least one untrainable species: 1038 of 1478
share of all species-detections that are untrainable: 0.311


## The scoring function

ROC-AUC is computed per species and then averaged, rather than pooled across all species at once. Pooling would let the common species dominate the number entirely, which given the avian skew in this dataset would hide exactly the failures worth knowing about.

One practical constraint drives the filtering below. ROC-AUC is undefined for a species that is either never present or always present in the segments being scored, because there is nothing to rank against. Those species are excluded from the average and the count of species actually scored is reported alongside it, so a high score computed over four species is never mistaken for a high score over two hundred.

Average precision is reported next to it. It behaves differently under heavy class imbalance, so a large gap between the two is itself informative.

In [ ]:
def scoreable(y_true):
    """Species with both positives and negatives present, so ROC-AUC is defined."""
    pos = y_true.sum(axis=0)
    n = y_true.shape[0]
    return (pos > 0) & (pos < n)


# Evaluate predictions against the truth matrix, returning a dictionary of metrics.
def evaluate(y_true, y_score, class_names):
    mask = scoreable(y_true)
    if mask.sum() == 0:
        return None

    yt = y_true[:, mask]
    ys = y_score[:, mask]
    names = np.array(class_names)[mask]

    auc = np.array([roc_auc_score(yt[:, i], ys[:, i]) for i in range(yt.shape[1])]) # array of per-species ROC-AUC scores
    ap = np.array([average_precision_score(yt[:, i], ys[:, i]) for i in range(yt.shape[1])]) # array of per-species average precision scores

# Create a DataFrame of per-species metrics, sorted by ROC-AUC
    per_species = pd.DataFrame({
        'species': names,
        'roc_auc': auc,
        'avg_precision': ap,
        'n_positive': yt.sum(axis=0),
    }).sort_values('roc_auc')


    return {
        'n_segments': int(y_true.shape[0]),
        'n_species_scored': int(mask.sum()),
        'macro_roc_auc': float(auc.mean()),
        'macro_avg_precision': float(ap.mean()),
        'per_species': per_species,
    }

## Testing the harness

The harness is checked against two sets of predictions whose correct score is already known. Random scores should give a macro ROC-AUC of roughly 0.5, since random ranking is no better than chance. Predictions copied directly from the truth should give exactly 1.0.

If either check fails, the harness is wrong, and any number it later reports about a real model would be meaningless.

In [ ]:
rng = np.random.default_rng(0) # for reproducibility

# Generate random scores and evaluate them
random_scores = rng.random(Y.shape)
res_random = evaluate(Y, random_scores, all_species)
print("random predictions")
print("  species scored:", res_random['n_species_scored'])
print("  macro ROC-AUC :", round(res_random['macro_roc_auc'], 4), " (expect ~0.5)")
print("  macro AP      :", round(res_random['macro_avg_precision'], 4))

perfect_scores = Y.astype(float)
res_perfect = evaluate(Y, perfect_scores, all_species)
print()
print("perfect predictions")
print("  macro ROC-AUC :", round(res_perfect['macro_roc_auc'], 4), " (expect 1.0)")

random predictions
  species scored: 75
  macro ROC-AUC : 0.5004  (expect ~0.5)
  macro AP      : 0.0627

perfect predictions
  macro ROC-AUC : 1.0  (expect 1.0)


## Breaking the result down by site

A single overall number hides the thing this project cares about. S22 holds roughly two thirds of the labelled segments, so an overall score is largely a score for S22. Reporting per site shows whether performance holds up at the sparsely sampled sites, and the segment count is carried alongside each score so a result resting on thirty segments is never read as equal to one resting on nine hundred.

In [ ]:
# Evaluate by site, using the same evaluate() function but filtering the data by site first
def evaluate_by_site(y_true, y_score, sites, class_names):
    rows = []
    for site in sorted(set(sites)):
        idx = [i for i, s in enumerate(sites) if s == site]
        r = evaluate(y_true[idx], y_score[idx], class_names)
        rows.append({
            'site': site,
            'n_segments': len(idx),
            'n_species_scored': r['n_species_scored'] if r else 0,
            'macro_roc_auc': round(r['macro_roc_auc'], 4) if r else np.nan,
            'macro_avg_precision': round(r['macro_avg_precision'], 4) if r else np.nan,
        })
    return pd.DataFrame(rows)


print(evaluate_by_site(Y, random_scores, sites, all_species))

  site  n_segments  n_species_scored  macro_roc_auc  macro_avg_precision
0  S03          48                 4         0.4896               0.6342
1  S08         120                19         0.4955               0.3110
2  S09          38                 5         0.3729               0.2337
3  S13          48                 6         0.5505               0.4926
4  S15          96                12         0.5314               0.2433
5  S18          30                 4         0.6104               0.4547
6  S19          72                17         0.5241               0.3465
7  S22         954                29         0.4814               0.1688
8  S23          72                13         0.4857               0.3151


## Breaking the result down by taxonomic group

The training data is overwhelmingly avian. Reporting performance by taxonomic group makes the consequence of that visible, rather than letting a strong bird score stand in for the whole model. Groups with almost no training data are expected to score poorly, and saying so with numbers is more useful than noting the imbalance in passing.

In [ ]:
# Map species to their respective groups (class_name)
group_map = dict(zip(tax['primary_label'].astype(str), tax['class_name']))

# Evaluate by group (class_name) using the group_map to map species to their respective groups
def evaluate_by_group(result, group_map):
    df = result['per_species'].copy()
    df['group'] = df['species'].map(group_map)
    return (df.groupby('group')
              .agg(n_species=('species', 'count'),
                   mean_roc_auc=('roc_auc', 'mean'),
                   mean_avg_precision=('avg_precision', 'mean'))
              .round(4))


print(evaluate_by_group(res_random, group_map))

          n_species  mean_roc_auc  mean_avg_precision
group                                                
Amphibia         17        0.5014              0.1702
Aves             28        0.4935              0.0305
Insecta          25        0.5046              0.0338
Mammalia          4        0.5072              0.0186
Reptilia          1        0.5444              0.0393


## Findings from the harness

The harness is verified: random predictions score a macro ROC-AUC of roughly 0.5, perfect predictions score 1.0.

The label space is 234 species, of which 206 have training audio. The remaining 28, comprising 25 insect sonotypes and three frogs, have none. All 28 appear in the field recordings.

Of the 75 species audible in the labelled soundscapes, 47 are trainable from focal clips and 28 are not. Those 28 appear in 1,038 of the 1,478 segments and account for 31% of all species-detections.

A classifier trained only on focal clips is therefore structurally unable to detect roughly a third of what the recorders captured, and the gap falls almost entirely on insects. This directly limits any species richness estimate derived from such a model, and it does so unevenly across sites depending on how insect-dominated each soundscape is.